# Tushare vs 米筐 数据验证 Notebook

## 目的
对比 Tushare (`ts_stock_all_data` / `ts_daily_basic`) 与 米筐 (`rq_stock_all_data`) 的数据，
核心验证 **自由流通市值** (Tushare: `free_share × close` vs 米筐: `mv_A_free_float`) 的一致性，
以及其他共有字段的对比。

## 验证时间范围
全部共同交易日：2021-01-04 ~ 2026-04-07 (共 1272 天)

In [25]:
import sys, os

# 路径修复：确保能从项目根目录导入 my_utils
project_root = os.path.dirname(os.path.abspath(os.getcwd()))
# 检查当前目录或上级目录是否包含 my_utils
for p in [os.getcwd(), os.path.dirname(os.getcwd()), project_root]:
    if os.path.isdir(os.path.join(p, 'my_utils')):
        if p not in sys.path:
            sys.path.insert(0, p)
        break

import polars as pl
import polars.selectors as cs
import numpy as np
from my_utils.fun import read_day_data, DATA_ROOT_DIR

START_DATE = '2021-01-04'
END_DATE = '2026-04-07'

print(f"验证时间范围: {START_DATE} ~ {END_DATE}")

print("正在读取 米筐 (RQ) 日线数据...")
rq_day = read_day_data(START_DATE, END_DATE, file_path='rq_stock_all_data')
print(f"  RQ 日线: {rq_day.height} 行, {len(rq_day.columns)} 列")

print("正在读取 Tushare (TS) 日线行情...")
ts_day = read_day_data(START_DATE, END_DATE, file_path='ts_stock_all_data')
print(f"  TS 日线行情: {ts_day.height} 行, {len(ts_day.columns)} 列")

print("正在读取 Tushare (TS) 每日指标 (daily_basic)...")
ts_basic = read_day_data(START_DATE, END_DATE, file_path='ts_daily_basic')
print(f"  TS 每日指标: {ts_basic.height} 行, {len(ts_basic.columns)} 列")

验证时间范围: 2021-01-04 ~ 2026-04-07
正在读取 米筐 (RQ) 日线数据...
  RQ 日线: 8132401 行, 20 列
正在读取 Tushare (TS) 日线行情...
  TS 日线行情: 6207841 行, 33 列
正在读取 Tushare (TS) 每日指标 (daily_basic)...
  TS 每日指标: 6207510 行, 18 列


## 1. 字段与 Schema 对比

In [26]:
rq_cols = set(rq_day.columns)
ts_cols = set(ts_day.columns)
tsb_cols = set(ts_basic.columns)

print("--- RQ 日线 vs TS 日线行情 ---")
common = rq_cols & ts_cols
only_rq = rq_cols - ts_cols
only_ts = ts_cols - rq_cols
print(f"共同列: {len(common)} 个: {sorted(common) if common else '无'}")
print(f"仅 RQ 有: {sorted(only_rq) if only_rq else '无'}")
print(f"仅 TS 日线有: {sorted(only_ts) if only_ts else '无'}")

print("\n--- RQ 日线 vs TS 每日指标 ---")
common_b = rq_cols & tsb_cols
only_rq_b = rq_cols - tsb_cols
only_tsb = tsb_cols - rq_cols
print(f"共同列: {len(common_b)} 个: {sorted(common_b) if common_b else '无'}")
print(f"仅 RQ 有: {sorted(only_rq_b) if only_rq_b else '无'}")
print(f"仅 TS 每日指标有: {sorted(only_tsb) if only_tsb else '无'}")

# Schema 类型对比
print("\n--- Schema 类型对比 (RQ vs TS日线) ---")
print(f"{'列名':<25} {'RQ类型':<20} {'TS类型':<20} {'匹配'}")
print("-" * 70)
for col in sorted(common):
    rq_dtype = rq_day[col].dtype
    ts_dtype = ts_day[col].dtype
    match = '✓' if str(rq_dtype) == str(ts_dtype) else '✗'
    print(f"{col:<25} {str(rq_dtype):<20} {str(ts_dtype):<20} {match}")

--- RQ 日线 vs TS 日线行情 ---
共同列: 14 个: ['amount', 'close', 'code', 'high', 'limit_down', 'limit_up', 'low', 'name', 'open', 'pct', 'pre_close', 'total_mv', 'trading_date', 'volume']
仅 RQ 有: ['adj_factor', 'circulation_mv', 'is_st', 'is_suspended', 'mv_A_free_float', 'turnover_rate']
仅 TS 日线有: ['activity', 'area', 'attack', 'avg_price', 'avg_turnover', 'buying', 'change', 'float_mv', 'float_share', 'industry', 'pe', 'selling', 'strength', 'swing', 'total_share', 'turn_over', 'type', 'type_name', 'vol_ratio']

--- RQ 日线 vs TS 每日指标 ---
共同列: 5 个: ['close', 'code', 'total_mv', 'trading_date', 'turnover_rate']
仅 RQ 有: ['adj_factor', 'amount', 'circulation_mv', 'high', 'is_st', 'is_suspended', 'limit_down', 'limit_up', 'low', 'mv_A_free_float', 'name', 'open', 'pct', 'pre_close', 'volume']
仅 TS 每日指标有: ['circ_mv', 'dv_ratio', 'dv_ttm', 'float_share', 'free_share', 'pb', 'pe', 'pe_ttm', 'ps', 'ps_ttm', 'total_share', 'turnover_rate_f', 'volume_ratio']

--- Schema 类型对比 (RQ vs TS日线) ---
列名          

## 2. 股票覆盖度对比

In [27]:
# 按交易日统计股票数
rq_daily_counts = rq_day.group_by('trading_date').agg(pl.len().alias('rq_count')).sort('trading_date')
ts_daily_counts = ts_day.group_by('trading_date').agg(pl.len().alias('ts_count')).sort('trading_date')

compare = rq_daily_counts.join(ts_daily_counts, on='trading_date', how='full').sort('trading_date')
compare = compare.with_columns(
    (pl.col('rq_count') / pl.col('ts_count') * 100).alias('coverage_pct')
)

print("每日股票覆盖度 (前10天 + 后10天):")
print(compare.head(10).to_pandas().to_string(index=False))
print("...")
print(compare.tail(10).to_pandas().to_string(index=False))

avg_cov = compare['coverage_pct'].mean()
print(f"\n平均覆盖度: {avg_cov:.1f}%")

# 代码级别对比
rq_codes = set(rq_day['code'].unique().to_list())
ts_code_set = set(ts_day['code'].unique().to_list())
print(f"\nRQ 总股票数: {len(rq_codes)}")
print(f"TS 总股票数: {len(ts_code_set)}")
print(f"交集: {len(rq_codes & ts_code_set)}")
print(f"仅 RQ: {len(rq_codes - ts_code_set)}")
print(f"仅 TS: {len(ts_code_set - rq_codes)}")

每日股票覆盖度 (前10天 + 后10天):
trading_date  rq_count trading_date_right  ts_count  coverage_pct
  2021-01-04      5650         2021-01-04      4120  137.13592233
  2021-01-05      5650         2021-01-05      4121  137.10264499
  2021-01-06      5653         2021-01-06      4123  137.10890129
  2021-01-07      5656         2021-01-07      4127  137.04870366
  2021-01-08      5657         2021-01-08      4128  137.03972868
  2021-01-11      5657         2021-01-11      4124  137.17264791
  2021-01-12      5659         2021-01-12      4125  137.18787879
  2021-01-13      5660         2021-01-13      4127  137.14562636
  2021-01-14      5660         2021-01-14      4124  137.24539282
  2021-01-15      5661         2021-01-15      4121  137.36957049
...
trading_date  rq_count trading_date_right  ts_count  coverage_pct
  2026-03-24      6680         2026-03-24      5198  128.51096576
  2026-03-25      6682         2026-03-25      5198  128.54944209
  2026-03-26      6682         2026-03-26      51

In [28]:
# 仅 RQ 有、TS 没有的股票列表
only_rq_codes = sorted(rq_codes - ts_code_set)
print(f"仅 RQ 有的股票: {len(only_rq_codes)} 只\n")

# 按交易所分组
szse = [c for c in only_rq_codes if c.startswith('SZSE')]
shse = [c for c in only_rq_codes if c.startswith('SHSE')]
print(f"  深交所(SZSE): {len(szse)} 只")
print(f"  上交所(SHSE): {len(shse)} 只")

# 取部分样本展示
print(f"\n--- 前30只仅RQ有的股票 ---")
print(only_rq_codes[:30])

# 取这些股票在 RQ 中的名称和最新交易日数据
only_rq_df = rq_day.filter(pl.col('code').is_in(only_rq_codes))
latest_date = only_rq_df.select(pl.col('trading_date').max()).item()
only_rq_latest = only_rq_df.filter(pl.col('trading_date') == latest_date)

print(f"\n--- 仅RQ有的股票 (最新交易日: {latest_date}) 示例 ---")
sample_cols = ['code', 'name', 'trading_date', 'close', 'total_mv', 'mv_A_free_float']
have_cols = [c for c in sample_cols if c in only_rq_latest.columns]
print(only_rq_latest.select(have_cols).head(20).to_pandas().to_string(index=False))

# 市值分布
if 'mv_A_free_float' in only_rq_latest.columns:
    mv = only_rq_latest['mv_A_free_float'].drop_nulls() / 1e8
    print(f"\n--- 仅RQ有的股票 自由流通市值分布 (亿元) ---")
    print(f"  数量: {mv.len()}")
    print(f"  最小值: {mv.min():.2f} 亿")
    print(f"  P25:   {mv.quantile(0.25):.2f} 亿")
    print(f"  中位数: {mv.median():.2f} 亿")
    print(f"  P75:   {mv.quantile(0.75):.2f} 亿")
    print(f"  最大值: {mv.max():.2f} 亿")
    print(f"  市值 < 10亿: {(mv < 10).sum() / mv.len():.1%}")
    print(f"  市值 < 20亿: {(mv < 20).sum() / mv.len():.1%}")

# 仅 TS 有的股票
only_ts_codes = sorted(ts_code_set - rq_codes)
print(f"\n--- 仅 TS 有的股票: {len(only_ts_codes)} 只 ---")
print(only_ts_codes if only_ts_codes else "无")

仅 RQ 有的股票: 1623 只

  深交所(SZSE): 372 只
  上交所(SHSE): 1251 只

--- 前30只仅RQ有的股票 ---
['SHSE.000001', 'SHSE.000002', 'SHSE.000003', 'SHSE.000004', 'SHSE.000005', 'SHSE.000006', 'SHSE.000007', 'SHSE.000008', 'SHSE.000009', 'SHSE.000010', 'SHSE.000011', 'SHSE.000012', 'SHSE.000013', 'SHSE.000015', 'SHSE.000016', 'SHSE.000017', 'SHSE.000018', 'SHSE.000019', 'SHSE.000020', 'SHSE.000021', 'SHSE.000022', 'SHSE.000025', 'SHSE.000026', 'SHSE.000027', 'SHSE.000028', 'SHSE.000029', 'SHSE.000030', 'SHSE.000031', 'SHSE.000032', 'SHSE.000033']

--- 仅RQ有的股票 (最新交易日: 2026-04-07) 示例 ---
       code name trading_date     close  total_mv  mv_A_free_float
SHSE.000001        2026-04-07 3890.1645       0.0              0.0
SHSE.000002        2026-04-07 4079.0935       0.0              0.0
SHSE.000003        2026-04-07  263.5791       0.0              0.0
SHSE.000004        2026-04-07 3650.4794       0.0              0.0
SHSE.000005        2026-04-07 2635.5238       0.0              0.0
SHSE.000006        2026-04-0

## 3. 核心：自由流通市值对比

**口径说明：**
- Tushare: `free_share`（万股，自由流通股本）× `close`（元/股）÷ 1e4 = **亿元**
- 米筐: `mv_A_free_float`（元，= free_circulation × close）÷ 1e8 = **亿元**

In [29]:
# === 核心：自由流通市值对比 ===

# 1. 计算 TS 自由流通市值 (亿元)
ts_ff = ts_basic.filter(
    pl.col('free_share').is_not_null() & pl.col('close').is_not_null() & (pl.col('free_share') > 0)
).with_columns(
    (pl.col('free_share') * pl.col('close') / 1e4).alias('ts_ff_mv')  # 亿元
).select(['code', 'trading_date', 'ts_ff_mv'])

print(f"TS 有 free_share 数据的行数: {ts_ff.height:,}")

# 2. 取 RQ 自由流通市值 (亿元)
rq_ff = rq_day.filter(
    pl.col('mv_A_free_float').is_not_null() & (pl.col('mv_A_free_float') > 0)
).with_columns(
    (pl.col('mv_A_free_float') / 1e8).alias('rq_ff_mv')  # 亿元
).select(['code', 'trading_date', 'rq_ff_mv'])

print(f"RQ 有 mv_A_free_float 数据的行数: {rq_ff.height:,}")

# 3. 合并对比
merged = ts_ff.join(rq_ff, on=['code', 'trading_date'], how='inner')
print(f"合并后有效行数: {merged.height:,}")

merged = merged.with_columns([
    (pl.col('ts_ff_mv') / pl.col('rq_ff_mv')).alias('ratio'),
    (pl.col('ts_ff_mv') - pl.col('rq_ff_mv')).alias('diff'),
])

# 4. 比值分布
print(f"\n=== TS自由流通/RQ自由流通 比值分布 ===")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    val = merged['ratio'].quantile(p/100)
    print(f"  P{p:2d}: {val:.4f}")

print(f"\n均值: {merged['ratio'].mean():.4f}")
print(f"TS > RQ 的占比: {merged.filter(pl.col('ratio') > 1).height / merged.height:.1%}")

# 5. 绝对差异分布 (亿元)
print(f"\n=== 绝对差异 (TS自由 - RQ自由) 分布 (亿元) ===")
for p in [1, 5, 25, 50, 75, 95, 99]:
    val = merged['diff'].quantile(p/100)
    print(f"  P{p:2d}: {val:.4f} 亿")

print(f"\n中位置信度: TS/RQ 中位数比值 = {merged['ratio'].median():.4f}")
print(f"-> {'TS口径 >= RQ口径' if merged['ratio'].median() >= 1 else 'RQ口径 >= TS口径'} (中位数)")

TS 有 free_share 数据的行数: 6,207,504
RQ 有 mv_A_free_float 数据的行数: 6,120,360
合并后有效行数: 6,104,543

=== TS自由流通/RQ自由流通 比值分布 ===
  P 1: 0.7799
  P 5: 0.9499
  P10: 0.9998
  P25: 1.0000
  P50: 1.0000
  P75: 1.0393
  P90: 1.1235
  P95: 1.2165
  P99: 1.5870

均值: 1.0400
TS > RQ 的占比: 59.8%

=== 绝对差异 (TS自由 - RQ自由) 分布 (亿元) ===
  P 1: -11.3134 亿
  P 5: -1.2297 亿
  P25: 0.0000 亿
  P50: 0.0000 亿
  P75: 1.3548 亿
  P95: 15.3844 亿
  P99: 83.3351 亿

中位置信度: TS/RQ 中位数比值 = 1.0000
-> TS口径 >= RQ口径 (中位数)


In [30]:
# 6. 逐股票中位比值
stock_ratio = merged.group_by('code').agg([
    pl.col('ratio').median().alias('median_ratio'),
    pl.col('ratio').count().alias('n_days'),
    pl.col('ts_ff_mv').median().alias('med_ts_ff'),
    pl.col('rq_ff_mv').median().alias('med_rq_ff'),
]).sort('median_ratio')

print(f"逐股票中位比值分布 ({stock_ratio.height} 只股票):")
medians = stock_ratio['median_ratio']
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p:2d}: {medians.quantile(p/100):.4f}")

print(f"\n中位比值 < 0.9 的股票数: {stock_ratio.filter(pl.col('median_ratio') < 0.9).height}")
print(f"中位比值 > 1.1 的股票数: {stock_ratio.filter(pl.col('median_ratio') > 1.1).height}")

# 7. 口径一致性判断
med = stock_ratio['median_ratio'].median()
print(f"\n结论: TS自由/RQ自由 股票级中位比值 = {med:.4f}")
if med > 1.1:
    print(f"  -> RQ 的自由流通口径显著严格于 Tushare (RQ值平均只有TS的 {100/med:.0f}%)")
elif med < 0.9:
    print(f"  -> TS 的自由流通口径显著严格于 米筐")
else:
    print(f"  -> 两个数据源的自由流通市值基本一致 (差异在10%以内)")

逐股票中位比值分布 (5385 只股票):
  P 1: 0.9069
  P 5: 1.0000
  P10: 1.0000
  P25: 1.0000
  P50: 1.0000
  P75: 1.0301
  P90: 1.0967
  P95: 1.1647
  P99: 1.4184

中位比值 < 0.9 的股票数: 50
中位比值 > 1.1 的股票数: 510

结论: TS自由/RQ自由 股票级中位比值 = 1.0000
  -> 两个数据源的自由流通市值基本一致 (差异在10%以内)


In [31]:
# 8. 差异最大的股票
print("--- TS >> RQ (RQ口径更严格) 的10只股票 ---")
top10 = stock_ratio.filter(pl.col('median_ratio') > 1).sort('median_ratio', descending=True).head(10)
print(top10.with_columns([
    (pl.col('med_ts_ff') - pl.col('med_rq_ff')).alias('diff_med')
]).to_pandas().to_string(index=False))

print(f"\n--- RQ >> TS (TS口径更严格) 的10只股票 ---")
bottom10 = stock_ratio.filter(pl.col('median_ratio') < 1).sort('median_ratio').head(10)
print(bottom10.with_columns([
    (pl.col('med_ts_ff') - pl.col('med_rq_ff')).alias('diff_med')
]).to_pandas().to_string(index=False))

# 9. 按市值分组看差异
merged_with_group = merged.with_columns([
    pl.when(pl.col('rq_ff_mv') < 10).then(pl.lit('小盘(<10亿)'))
    .when(pl.col('rq_ff_mv') < 50).then(pl.lit('中盘(10-50亿)'))
    .when(pl.col('rq_ff_mv') < 200).then(pl.lit('大盘(50-200亿)'))
    .otherwise(pl.lit('超大(>200亿)')).alias('mkt_cap_group')
])

print(f"\n=== 按市值分组的中位比值 ===")
for group in ['小盘(<10亿)', '中盘(10-50亿)', '大盘(50-200亿)', '超大(>200亿)']:
    g = merged_with_group.filter(pl.col('mkt_cap_group') == group)
    if g.height > 0:
        print(f"  {group}: {g.height:,}行, 中位比值={g['ratio'].median():.4f}")

--- TS >> RQ (RQ口径更严格) 的10只股票 ---
       code  median_ratio  n_days     med_ts_ff    med_rq_ff     diff_med
SHSE.689009   10.46041912    1252  135.87321738  13.06508616 122.80813122
SHSE.600337    2.97279693    1242   23.26291125   8.01389366  15.24901759
SHSE.600400    2.88814451    1252   23.35630200   8.03607820  15.32022381
SHSE.600236    2.67585750    1252   71.11796464  26.50343850  44.61452614
SHSE.601816    2.41692901    1252 1247.49438410 525.34402682 722.15035728
SHSE.603315    2.33529649    1241   19.40354918   8.13503015  11.26851903
SZSE.002259    2.30706743    1250   22.53909262  11.08730538  11.45178724
SZSE.000932    2.24946513    1252  194.65534001  87.15887253 107.49646748
SHSE.600383    2.19849801    1252  290.77199512 138.22383633 152.54815879
SHSE.600687    2.18525118      30    2.01041214   0.91999133   1.09042081

--- RQ >> TS (TS口径更严格) 的10只股票 ---
       code  median_ratio  n_days    med_ts_ff    med_rq_ff      diff_med
SHSE.603650    0.34406978    1252  59.58196

## 4. 其他字段对比：价格 & 量

In [32]:
# ---- 4.1 收盘价对比 ----
price_merged = ts_day.select(['code', 'trading_date', 'close']).join(
    rq_day.select(['code', 'trading_date', 'close']),
    on=['code', 'trading_date'], suffix='_rq', how='inner'
)

close_diff = (price_merged['close'] - price_merged['close_rq']).abs()
rel_close_diff = (close_diff / price_merged['close'].abs()).filter(price_merged['close'].abs() > 0)
print(f"--- 收盘价 (close) 对比 ---")
print(f"  匹配行数: {price_merged.height:,}")
print(f"  最大绝对差: {close_diff.max():.6f}")
print(f"  中位绝对差: {close_diff.median():.10f}")
print(f"  均值相对差: {rel_close_diff.mean():.6e}")
print(f"  完全相同(差=0)的比例: {(close_diff == 0).sum() / close_diff.len():.1%}")
print(f"  差>0.01的比例: {(close_diff > 0.01).sum() / close_diff.len():.1%}")

# ---- 4.2 涨跌幅对比 ----
pct_merged = ts_day.select(['code', 'trading_date', 'pct']).join(
    rq_day.select(['code', 'trading_date', 'pct']),
    on=['code', 'trading_date'], suffix='_rq', how='inner'
)
pct_diff = (pct_merged['pct'] - pct_merged['pct_rq']).abs()
print(f"\n--- 涨跌幅 (pct) 对比 ---")
print(f"  最大绝对差: {pct_diff.max():.6f}%")
print(f"  中位绝对差: {pct_diff.median():.6f}%")
print(f"  完全相同比例: {(pct_diff == 0).sum() / pct_diff.len():.1%}")

# ---- 4.3 成交量对比 ----
# 注意：TS volume = 手(1手=100股)，RQ volume = 股，需要对齐
vol_merged = ts_day.select(['code', 'trading_date',
    (pl.col('volume') * 100).alias('volume_shares')  # TS: 手→股
]).join(
    rq_day.select(['code', 'trading_date', 'volume']),
    on=['code', 'trading_date'], suffix='_rq', how='inner'
)
vol_diff = (vol_merged['volume_shares'] - vol_merged['volume']).abs()
rel_vol_diff = (vol_diff / vol_merged['volume_shares'].abs()).filter(vol_merged['volume_shares'].abs() > 0)
print(f"\n--- 成交量 (volume) 对比 (TS手→股对齐后) ---")
print(f"  匹配行数: {vol_merged.height:,}")
print(f"  最大绝对差: {vol_diff.max():.2e}")
print(f"  中位相对差: {rel_vol_diff.median():.6f}")
print(f"  中位TS量(股): {vol_merged['volume_shares'].median():.2e}")
print(f"  中位RQ量(股): {vol_merged['volume'].median():.2e}")

--- 收盘价 (close) 对比 ---
  匹配行数: 6,207,363
  最大绝对差: 0.000000
  中位绝对差: 0.0000000000
  均值相对差: 0.000000e+00
  完全相同(差=0)的比例: 100.0%
  差>0.01的比例: 0.0%

--- 涨跌幅 (pct) 对比 ---
  最大绝对差: 0.048595%
  中位绝对差: 0.000026%
  完全相同比例: 2.8%

--- 成交量 (volume) 对比 (TS手→股对齐后) ---
  匹配行数: 6,207,363
  最大绝对差: 1.60e+02
  中位相对差: 0.000000
  中位TS量(股): 7.05e+06
  中位RQ量(股): 7.05e+06


## 5. 其他字段对比：市值 & 换手率

In [33]:
# ---- 5.1 总市值对比 ----
# TS total_mv 在 ts_basic 中（万元），RQ total_mv 在 rq_day 中（元）
ts_tmv = ts_basic.select(['code', 'trading_date', 'total_mv'])
rq_tmv = rq_day.select(['code', 'trading_date', 'total_mv'])

tmv_merged = ts_tmv.join(rq_tmv, on=['code', 'trading_date'], how='inner', suffix='_rq')
# 统一为亿元
tmv_merged = tmv_merged.with_columns([
    (pl.col('total_mv') / 1e4).alias('ts_mv_yi'),     # TS 万元→亿元
    (pl.col('total_mv_rq') / 1e8).alias('rq_mv_yi'),   # RQ 元→亿元
])
tmv_merged = tmv_merged.with_columns(
    (pl.col('ts_mv_yi') / pl.col('rq_mv_yi').clip(lower_bound=1)).alias('mv_ratio')
)

print(f"--- 总市值对比 (TS:万元, RQ:元, 统一为亿元) ---")
print(f"  匹配行数: {tmv_merged.height:,}")
tmv_valid = tmv_merged.filter(pl.col('rq_mv_yi') > 0)
print(f"  TS/RQ 中位比值: {tmv_valid['mv_ratio'].median():.4f}")
print(f"  TS 总市值中位数: {tmv_valid['ts_mv_yi'].median():.2f} 亿")
print(f"  RQ 总市值中位数: {tmv_valid['rq_mv_yi'].median():.2f} 亿")

# ---- 5.2 换手率对比 ----
tr_merged = ts_basic.select(['code', 'trading_date', 'turnover_rate']).join(
    rq_day.select(['code', 'trading_date', 'turnover_rate']),
    on=['code', 'trading_date'], how='inner', suffix='_rq'
)
print(f"\n--- 换手率 (turnover_rate) 对比 ---")
print(f"  匹配行数: {tr_merged.height:,}")
tr_diff = (tr_merged['turnover_rate'] - tr_merged['turnover_rate_rq']).abs()
print(f"  最大绝对差: {tr_diff.max():.4f}%")
print(f"  中位绝对差: {tr_diff.median():.4f}%")
print(f"  中位TS换手率: {tr_merged['turnover_rate'].median():.4f}%")
print(f"  中位RQ换手率: {tr_merged['turnover_rate_rq'].median():.4f}%")

--- 总市值对比 (TS:万元, RQ:元, 统一为亿元) ---
  匹配行数: 6,207,510
  TS/RQ 中位比值: 1.0000
  TS 总市值中位数: 56.52 亿
  RQ 总市值中位数: 56.16 亿

--- 换手率 (turnover_rate) 对比 ---
  匹配行数: 6,207,510
  最大绝对差: 107.5746%
  中位绝对差: 0.0000%
  中位TS换手率: 1.7164%
  中位RQ换手率: 1.6555%


## 6. 补充：流通市值对比 (TS float_mv vs RQ circulation_mv)

In [34]:
# TS 流通市值 = float_mv (ts_stock_all_data, 亿元), RQ 流通市值 = circulation_mv (rq_day, 元)
cirk_merged = ts_day.select(['code', 'trading_date', 'float_mv']).join(
    rq_day.select(['code', 'trading_date', 'circulation_mv']),
    on=['code', 'trading_date'], how='inner', suffix='_rq'
)
cirk_merged = cirk_merged.with_columns(
    (pl.col('circulation_mv') / 1e8).alias('rq_cirk_yi')  # 元→亿元
)

print(f"--- 流通市值对比 ---")
print(f"  匹配行数: {cirk_merged.height:,}")

cirk_ratio = (cirk_merged['float_mv'] / cirk_merged['rq_cirk_yi'].clip(lower_bound=1))
print(f"  TS(float_mv)/RQ(circulation_mv) 中位比值: {cirk_ratio.median():.4f}")

# 自由流通 vs 流通的对比
ff_cirk_merged = merged.select(['code', 'trading_date', 'ts_ff_mv', 'rq_ff_mv']).join(
    cirk_merged.select(['code', 'trading_date', 'float_mv']),
    on=['code', 'trading_date'], how='inner'
)
print(f"\n  自由流通/流通 比值:")
print(f"  TS自由/TS流通: median={(ff_cirk_merged['ts_ff_mv'] / ff_cirk_merged['float_mv'].clip(lower_bound=1)).median():.4f}")

rq_free_cirk = merged.select(['code', 'trading_date', 'rq_ff_mv']).join(
    rq_day.select(['code', 'trading_date', 'circulation_mv']),
    on=['code', 'trading_date'], how='inner'
)
print(f"  RQ自由/RQ流通: median={(rq_free_cirk['rq_ff_mv'] / (rq_free_cirk['circulation_mv']/1e8).clip(lower_bound=1)).median():.4f}")

--- 流通市值对比 ---
  匹配行数: 6,207,363
  TS(float_mv)/RQ(circulation_mv) 中位比值: 1.0000

  自由流通/流通 比值:
  TS自由/TS流通: median=0.6599
  RQ自由/RQ流通: median=0.6422


## 7. 缺失值检查

In [35]:
print(f"{'列名':<25} {'RQ缺失数':<15} {'RQ缺失率':<15} {'TS缺失数':<15} {'TS缺失率'}")
print("-" * 85)

# RQ vs TS day common columns
for col in sorted(common):
    rq_null = rq_day[col].null_count()
    ts_null = ts_day[col].null_count()
    rq_rate = rq_null / rq_day.height * 100
    ts_rate = ts_null / ts_day.height * 100
    flag = ' <-- 注意' if abs(rq_rate - ts_rate) > 5 else ''
    print(f"{col:<25} {rq_null:<15} {rq_rate:<14.2f}% {ts_null:<15} {ts_rate:<10.2f}%{flag}")

# ts_basic 关键字段缺失
print(f"\n--- TS daily_basic 关键字段缺失 ---")
for col in ['free_share', 'close', 'total_mv', 'turnover_rate']:
    n = ts_basic[col].null_count()
    r = n / ts_basic.height * 100
    print(f"  {col:<20}: {n:>8,} / {ts_basic.height:,} ({r:.2f}%)")

print(f"\n--- RQ 关键字段缺失 ---")
for col in ['mv_A_free_float', 'circulation_mv', 'total_mv', 'turnover_rate']:
    n = rq_day[col].null_count()
    r = n / rq_day.height * 100
    print(f"  {col:<20}: {n:>8,} / {rq_day.height:,} ({r:.2f}%)")

列名                        RQ缺失数           RQ缺失率           TS缺失数           TS缺失率
-------------------------------------------------------------------------------------
amount                    0               0.00          % 0               0.00      %
close                     0               0.00          % 0               0.00      %
code                      0               0.00          % 0               0.00      %
high                      0               0.00          % 0               0.00      %
limit_down                0               0.00          % 478             0.01      %
limit_up                  0               0.00          % 478             0.01      %
low                       0               0.00          % 0               0.00      %
name                      0               0.00          % 0               0.00      %
open                      0               0.00          % 0               0.00      %
pct                       0               0.00          % 0 

## 8. 数据时效性 & 总结

### 数据时效性
- RQ (米筐): 更新至 **2026-05-20**
- TS (Tushare): 更新至 **2026-04-07**
- RQ 比 TS 多约 1.5 个月的最新数据

### 核心结论

查看以上各节输出，关注：

1. **自由流通市值一致性** (核心)
   - TS自由/RQ自由 比值分布中位数
   - 逐股票中位比值分析
   - 不同市值分组的差异
   
2. **其他字段一致性**
   - close/pct/volume 等基础行情字段
   - total_mv/turnover_rate 等指标字段

3. **覆盖度 & 数据时效**
   - RQ 的股票覆盖范围
   - RQ 的数据更新时效优势